In [1]:
import pickle
import pandas as pd

In [2]:
path = f'prepared_data/dataset_for_feature_engineering.pickle'

with open(path, 'rb') as f:
    df = pickle.load(f)

In [3]:
df.shape

(144900, 7)

In [4]:
df["date"] = pd.to_datetime(df["date"])

In [5]:
df = df.sort_values(["ticker","date"]).reset_index(drop=True)

In [7]:
df.describe()

,date,open,high,low,close,volume
count,144900,144900.000000,144900.000000,144900.000000,144900.000000,1.449000e+05
mean,2020-09-24 05:06:12.968944384,3464.369568,3508.167564,3415.176101,3464.172141,1.139041e+09
min,2013-03-25 00:00:00,0.003765,0.003965,0.003440,0.003775,1.000000e+00
25%,2017-09-20 00:00:00,39.600000,40.180000,38.955750,39.550000,2.076075e+05
50%,2021-02-10 00:00:00,150.285000,152.900000,148.000000,150.200000,1.848910e+06
75%,2023-12-07 00:00:00,633.200000,643.000000,623.525000,632.900000,1.574113e+07
max,2026-03-06 00:00:00,231000.000000,235700.000000,220100.000000,231000.000000,7.032681e+11
std,NaN,19565.247906,19806.728969,19294.166206,19567.499320,1.101455e+10


In [10]:
# Доходность — это процентное изменение цены акции за выбранный период времени.
# Она показывает, насколько выросла или упала цена по сравнению с прошлым значением.

# Экономический смысл:
# Если доходность положительная — акция растёт.
# Если отрицательная — акция падает.
# Чем больше модуль доходности, тем сильнее движение цены.

df["ret_1"]  = df.groupby("ticker")["close"].pct_change(1)   # дневное изменение цены
df["ret_3"]  = df.groupby("ticker")["close"].pct_change(1)   # изменение цены за три дня
df["ret_5"]  = df.groupby("ticker")["close"].pct_change(5)   # изменение за торговую неделю
df["ret_20"] = df.groupby("ticker")["close"].pct_change(20)  # изменение за торговый месяц

In [13]:
# Волатильность показывает, насколько нестабильно ведёт себя цена акции.
# Чем выше волатильность — тем выше риск и тем более резкие движения цены происходят.
#
# Экономический смысл:
# Высокая волатильность означает нервный, неустойчивый рынок.
# Низкая — спокойное, стабильное движение.

df["vol3"]  = df.groupby("ticker")["ret_1"].rolling(3).std().reset_index(0, drop=True)   # волатильность за 3 дня
df["vol_5"]  = df.groupby("ticker")["ret_1"].rolling(5).std().reset_index(0, drop=True)   # волатильность за 5 дней
df["vol_15"] = df.groupby("ticker")["ret_1"].rolling(15).std().reset_index(0, drop=True) # волатильность за 15 дней
df["vol_30"] = df.groupby("ticker")["ret_1"].rolling(30).std().reset_index(0, drop=True) # волатильность за 30 дней
df["vol_45"] = df.groupby("ticker")["ret_1"].rolling(45).std().reset_index(0, drop=True) # волатильность за 45 дней
df["vol_90"] = df.groupby("ticker")["ret_1"].rolling(90).std().reset_index(0, drop=True) # волатильность за 90 дней

In [16]:
# Моментум показывает, насколько сильно и в каком направлении двигалась цена акции
# за выбранный период времени.
#
# Экономический смысл:
# Положительный моментум означает, что акция находится в фазе роста.
# Отрицательный — что акция находится в фазе падения.

df["mom_3"]  = df.groupby("ticker")["close"].pct_change(3)   # сила движения за 3 дня
df["mom_5"]  = df.groupby("ticker")["close"].pct_change(5)   # сила движения за 5 дней
df["mom_15"]  = df.groupby("ticker")["close"].pct_change(15)   # сила движения за 5 дней
df["mom_20"] = df.groupby("ticker")["close"].pct_change(20)  # сила движения за 20 дней

In [19]:
# Скользящие средние показывают сглаженное направление движения цены акции.
# Они позволяют отделить устойчивый тренд от случайных рыночных колебаний.
#
# Экономический смысл:
# Если короткая средняя выше длинной — тренд восходящий.
# Если ниже — нисходящий.

df["sma_3"]  = df.groupby("ticker")["close"].transform(lambda x: x.rolling(3).mean())    # средняя цена за 3 дня
df["sma_5"]  = df.groupby("ticker")["close"].transform(lambda x: x.rolling(5).mean())    # средняя цена за 5 дней
df["sma_10"]  = df.groupby("ticker")["close"].transform(lambda x: x.rolling(10).mean())    # средняя цена за 5 дней
df["sma_20"] = df.groupby("ticker")["close"].transform(lambda x: x.rolling(20).mean())   # средняя цена за 20 дней

df["sma_ratio"] = df["sma_5"] / df["sma_20"] - 1   # относительное положение краткой и длинной средних


In [21]:
# Объём торгов показывает активность участников рынка.
# Он отражает, насколько активно инвесторы покупают и продают акцию.
#
# Экономический смысл:
# Рост объёма подтверждает силу движения цены.
# Падение объёма — признак ослабления интереса к акции.

df["vol_mean_5"] = df.groupby("ticker")["volume"].transform(lambda x: x.rolling(5).mean())  # средний объём за 5 дней
df["vol_mean_15"] = df.groupby("ticker")["volume"].transform(lambda x: x.rolling(15).mean())  # средний объём за 15 дней
df["vol_mean_45"] = df.groupby("ticker")["volume"].transform(lambda x: x.rolling(45).mean())  # средний объём за 45 дней
df["vol_ratio"]  = df["volume"] / df["vol_mean_5"]                                          # относительный объём


In [22]:
# Свечной диапазон показывает внутридневную амплитуду колебаний цены.
# Он отражает уровень рыночной неопределённости и борьбу покупателей и продавцов.

df["hl_range"] = (df["high"] - df["low"]) / df["close"]   # относительный дневной диапазон


In [23]:
# Временные признаки отражают календарную структуру торговых сессий.
# Рынок ведёт себя по-разному в разные дни недели и месяцы года из-за поведения инвесторов,
# налоговых периодов, отчётностей и ребалансировок фондов.

df["dow"] = df["date"].dt.weekday       # день недели (0 = понедельник, 4 = пятница)
df["month"] = df["date"].dt.month       # номер месяца (1–12)
df["week"] = df["date"].dt.isocalendar().week.astype(int)   # номер недели года
df["is_month_end"] = df["date"].dt.is_month_end.astype(int) # конец месяца (1 = да)
df["is_month_start"] = df["date"].dt.is_month_start.astype(int) # начало месяца

In [24]:
# Целевая переменная показывает, вырастет ли цена акции на следующий торговый день.
# 1 — если завтрашняя цена закрытия выше сегодняшней.
# 0 — если цена не выросла.

df["target"] = (df.groupby("ticker")["close"].shift(-1) > df["close"]).astype(int)

In [25]:
df["target_diff"] = (df.groupby("ticker")["close"].shift(-1) - df["close"])/df["close"]*100

In [26]:
import numpy as np

df["target_diff"].describe(np.arange(0,1,0.1))

count    144831.000000
mean          3.334378
std        1225.552276
min         -99.050944
0%          -99.050944
10%          -2.134974
20%          -1.244444
30%          -0.723888
40%          -0.339233
50%          -0.013108
60%           0.295068
70%           0.701754
80%           1.280956
90%           2.301587
max      466282.338184
Name: target_diff, dtype: float64

In [32]:
df[(df.ticker=='VTBR') & (df.date>'2024-07-01')].sort_values(by=['date']).head(20)

,date,open,high,low,close,volume,ticker,ret_1,ret_3,ret_5,...,vol_mean_45,vol_ratio,hl_range,dow,month,week,is_month_end,is_month_start,target,target_diff
143770,2024-07-02,0.02132,0.021365,0.021000,0.021010,56250780000,VTBR,-0.014078,-0.014078,0.026380,...,1.303674e+11,0.252602,0.017373,1,7,27,0,0,0,-3.069967
143771,2024-07-03,0.02100,0.021185,0.020260,0.020365,118577570000,VTBR,-0.030700,-0.030700,-0.059353,...,1.302698e+11,0.728881,0.045421,2,7,27,0,0,1,0.687454
143772,2024-07-04,0.02037,0.020950,0.020255,0.020505,145494940000,VTBR,0.006875,0.006875,-0.035513,...,1.322216e+11,1.082274,0.033894,3,7,27,0,0,0,-1.146062
143773,2024-07-05,0.02059,0.020790,0.020120,0.020270,93540060000,VTBR,-0.011461,-0.011461,-0.041154,...,1.336791e+11,0.796576,0.033054,4,7,27,0,0,0,-1.677356
143774,2024-07-08,0.02030,0.020570,0.019760,0.019930,87537340000,VTBR,-0.016774,-0.016774,-0.064758,...,1.346204e+11,0.872928,0.040642,0,7,28,0,0,1,466282.338184
143775,2024-07-15,100.00000,100.050000,91.650000,92.950000,97984027,VTBR,4662.823382,4662.823382,4423.083770,...,1.334166e+11,0.001100,0.090371,0,7,29,0,0,1,2.549758
143776,2024-07-16,92.93000,95.400000,91.800000,95.320000,49545775,VTBR,0.025498,0.025498,4679.579425,...,1.321701e+11,0.000758,0.037768,1,7,29,0,0,0,-1.678556
143777,2024-07-17,95.60000,96.690000,93.100000,93.720000,33479366,VTBR,-0.016786,-0.016786,4569.592538,...,1.312578e+11,0.000924,0.038306,2,7,29,0,0,1,2.315408
143778,2024-07-18,93.59000,96.150000,92.800000,95.890000,27363977,VTBR,0.023154,0.023154,4729.636408,...,1.305749e+11,0.001559,0.034936,3,7,29,0,0,1,0.458859
143779,2024-07-19,95.89000,97.660000,95.000000,96.330000,25117590,VTBR,0.004589,0.004589,4832.416959,...,1.300957e+11,0.537871,0.027613,4,7,29,0,0,1,3.550296


In [28]:
df.sort_values(by=["target_diff"], ascending=False)

,date,open,high,low,close,volume,ticker,ret_1,ret_3,ret_5,...,vol_mean_45,vol_ratio,hl_range,dow,month,week,is_month_end,is_month_start,target,target_diff
143774,2024-07-08,0.020300,0.02057,0.019760,0.01993,87537340000,VTBR,-0.016774,-0.016774,-0.064758,...,1.346204e+11,0.872928,0.040642,0,7,28,0,0,1,466282.338184
41633,2014-12-18,0.006895,0.00720,0.006301,0.00712,11040400000,IRAO,0.062845,0.062845,-0.117720,...,9.188698e+09,0.975288,0.126264,3,12,51,0,0,1,10631.741573
83658,2013-08-14,623.900000,623.90000,623.900000,623.90000,2,PHOR,NaN,NaN,NaN,...,NaN,NaN,0.000000,2,8,33,0,0,1,119.874980
134,2014-12-17,5.980000,6.90000,5.720000,6.61000,23976100,AFKS,0.099834,0.099834,-0.053009,...,1.842869e+07,0.644430,0.178517,2,12,51,0,0,1,106.202723
116538,2016-02-19,5.050000,5.25000,5.050000,5.16000,25700,SELG,0.021782,0.021782,0.025845,...,7.136689e+05,0.798137,0.038760,4,2,7,0,0,1,99.806202
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140519,2026-03-06,304.750000,305.35000,298.350000,300.20000,1268675,VKCO,-0.012500,-0.012500,-0.039514,...,3.226894e+06,0.708576,0.023318,4,3,10,0,0,0,NaN
140944,2026-03-06,79.850000,81.68000,78.930000,81.09000,1460593,VSEH,0.015529,0.015529,0.004584,...,1.360660e+06,1.460162,0.033913,4,3,10,0,0,0,NaN
144193,2026-03-06,85.850000,86.68000,85.575000,86.02000,39722553,VTBR,0.001047,0.001047,-0.010411,...,1.068790e+08,0.587636,0.012846,4,3,10,0,0,0,NaN
144487,2026-03-06,2414.000000,2415.00000,2390.500000,2395.00000,554503,X5,-0.005812,-0.005812,-0.014200,...,9.725366e+05,0.836701,0.010230,4,3,10,0,0,0,NaN


In [40]:
info = ['date', 'ticker']

In [41]:
df.drop(columns=info).corr()['target'].sort_values()

vol_ratio        -0.008652
is_month_start   -0.007215
month            -0.006843
volume           -0.006001
week             -0.005209
vol_mean_5       -0.004427
vol_mean_45      -0.002843
vol_mean_15      -0.002250
vol_90           -0.001502
hl_range         -0.000596
vol_15           -0.000368
mom_15           -0.000353
sma_20            0.000054
close             0.000077
low               0.000147
sma_10            0.000164
high              0.000174
open              0.000246
sma_3             0.000316
sma_5             0.000364
vol_45            0.000914
mom_20            0.001369
ret_20            0.001369
vol_30            0.001408
mom_3             0.001621
vol3              0.001681
ret_3             0.002726
ret_1             0.002726
sma_ratio         0.003019
mom_5             0.003642
ret_5             0.003642
vol_5             0.003673
is_month_end      0.007059
dow               0.013445
target            1.000000
Name: target, dtype: float64

In [42]:
features = df.drop(columns=info+['target']).columns

In [43]:
df[df['ret_1']>1][['ticker', 'date', "close",'ret_1']]

,ticker,date,close,ret_1
135,AFKS,2014-12-18,13.6300,1.062027
41634,IRAO,2015-01-20,0.7641,106.317416
83659,PHOR,2013-08-15,1371.8000,1.198750
143775,VTBR,2024-07-15,92.9500,4662.823382


In [44]:
def smooth_outliers(df, features, lower_percentile=0.1, upper_percentile=0.9):
    """
    Функция для сглаживания выбросов в заданных фичах.
    Заменяет значения, которые выходят за пределы интервала между нижним и верхним персентилем.
    
    :param df: pandas DataFrame, исходный датасет.
    :param features: список названий колонок (фич), в которых нужно проверить выбросы.
    :param lower_percentile: нижний персентиль, по умолчанию 0.1 (10-й персентиль).
    :param upper_percentile: верхний персентиль, по умолчанию 0.9 (90-й персентиль).
    :return: pandas DataFrame с заменёнными выбросами.
    """
    
    data = df.copy()
    
    for feature in features:
        if feature in data.columns:
            # Вычисляем персентиль для нижней и верхней границы
            lower_value = data[feature].quantile(lower_percentile)
            upper_value = data[feature].quantile(upper_percentile)
            
            # Заменяем значения, выходящие за пределы интервала
            data[feature] = data[feature].apply(lambda x: min(max(x, lower_value), upper_value))
    
    return data

In [45]:
df = smooth_outliers(df, features, lower_percentile=0.01, upper_percentile=0.99)

In [46]:
df[df['ret_1']>1][['ticker', 'date', "close",'ret_1']]

,ticker,date,close,ret_1


In [47]:
path = f'prepared_data/dataset_for_feature_selection.pickle'

samples = {
    'df'           : df,
    'info_fields'  : info,
    'features'     : features,
    'cat_features' : [],
    'num_features' : features,
    'target'       : 'target',
}
with open(path, 'wb') as f:
    pickle.dump(samples, f)

In [48]:
df.shape

(144900, 37)